In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit

PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)

mart = pd.read_parquet(
    PROCESSED / "mart_manufacturer_drug_month_monitoring.parquet"
)

mart["event_month"] = pd.to_datetime(mart["event_month"])
mart = mart.sort_values(
    ["manufacturer", "drug_name", "event_month"]
).reset_index(drop=True)

mart.shape

(1362400, 24)

In [2]:
keys = ["manufacturer", "drug_name"]

mart["events_next_month"] = (
    mart.groupby(keys)["events_count"]
    .shift(-1)
)

mart["target_next_month"] = (
    mart["events_next_month"]
    .gt(0)
    .where(mart["events_next_month"].notna())
    .astype("Int64")
)

mart[
    [
        "manufacturer",
        "drug_name",
        "event_month",
        "events_count",
        "events_next_month",
        "target_next_month",
    ]
].tail(10)

,manufacturer,drug_name,event_month,events_count,events_next_month,target_next_month
1362390,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-01-01,0.0,0.0,0
1362391,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-02-01,0.0,0.0,0
1362392,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-03-01,0.0,0.0,0
1362393,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-04-01,0.0,0.0,0
1362394,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-05-01,0.0,0.0,0
1362395,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-06-01,0.0,0.0,0
1362396,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-07-01,0.0,0.0,0
1362397,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-08-01,0.0,0.0,0
1362398,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-09-01,0.0,0.0,0
1362399,ярославская фармфабрика зао,элеутерококка экстракт жидкий,2026-10-01,0.0,NaN,<NA>


In [3]:
feature_columns = [
    "events_count",
    "batches_count",
    "letters_count",
    "events_prev_month",
    "events_change",
    "is_spike",
]

target_column = "target_next_month"

model_data = mart[
    [
        "manufacturer",
        "drug_name",
        "event_month",
    ]
    + feature_columns
    + [target_column]
].dropna()

model_data[target_column] = model_data[target_column].astype(int)

X = model_data[feature_columns]
y = model_data[target_column]

print("Строк:", len(model_data))
print("Доля положительного класса:", y.mean().round(4))
model_data.head()

Строк: 1352000
Доля положительного класса: 0.0111


,manufacturer,drug_name,event_month,events_count,batches_count,letters_count,events_prev_month,events_change,is_spike,target_next_month
1,,аримидекс,2005-02-01,0.0,0.0,0.0,0.0,0.0,0,0
2,,аримидекс,2005-03-01,0.0,0.0,0.0,0.0,0.0,0,0
3,,аримидекс,2005-04-01,0.0,0.0,0.0,0.0,0.0,0,0
4,,аримидекс,2005-05-01,0.0,0.0,0.0,0.0,0.0,0,0
5,,аримидекс,2005-06-01,0.0,0.0,0.0,0.0,0.0,0,0


In [4]:
model_data = model_data.sort_values("event_month").reset_index(drop=True)

X = model_data[feature_columns]
y = model_data[target_column]

split_index = int(len(model_data) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

train_end = model_data.iloc[split_index - 1]["event_month"]
test_start = model_data.iloc[split_index]["event_month"]

print("Train:", X_train.shape, "до", train_end.strftime("%Y-%m"))
print("Test:", X_test.shape, "с", test_start.strftime("%Y-%m"))
print("Доля событий в train:", y_train.mean().round(4))
print("Доля событий в test:", y_test.mean().round(4))

Train: (1081600, 6) до 2022-05
Test: (270400, 6) с 2022-06
Доля событий в train: 0.0131
Доля событий в test: 0.003


In [5]:
baseline_prediction = (X_test["events_count"] > 0).astype(int)

print(classification_report(
    y_test,
    baseline_prediction,
    target_names=["Нет события", "Есть событие"],
    digits=3,
))

              precision    recall  f1-score   support

 Нет события      0.998     0.998     0.998    269586
Есть событие      0.251     0.254     0.253       814

    accuracy                          0.995    270400
   macro avg      0.624     0.626     0.625    270400
weighted avg      0.996     0.995     0.995    270400



In [6]:
model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    random_state=42,
)

model.fit(X_train, y_train)

probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.5).astype(int)

print(classification_report(
    y_test,
    prediction,
    target_names=["Нет события", "Есть событие"],
    digits=3,
))

              precision    recall  f1-score   support

 Нет события      0.997     1.000     0.999    269586
Есть событие      0.598     0.060     0.109       814

    accuracy                          0.997    270400
   macro avg      0.797     0.530     0.554    270400
weighted avg      0.996     0.997     0.996    270400



In [7]:
auc = roc_auc_score(y_test, probability)

print(f"ROC-AUC: {auc:.3f}")

ROC-AUC: 0.681


In [8]:
test_data = model_data.iloc[split_index:].copy()
test_data["probability"] = probability
test_data["baseline_prediction"] = baseline_prediction

def precision_at_k(data, score_column, k=20):
    top = data.nlargest(k, score_column)
    return top[target_column].mean()

results = pd.DataFrame(
    {
        "Метод": [
            "Baseline: текущие события",
            "ML-модель",
        ],
        "Precision@20": [
            precision_at_k(test_data, "events_count"),
            precision_at_k(test_data, "probability"),
        ],
    }
)

results["Lift_vs_random"] = (
    results["Precision@20"] / y_test.mean()
)

results.round(4)

,Метод,Precision@20,Lift_vs_random
0,Baseline: текущие события,0.4,132.8747
1,ML-модель,0.6,199.3120


In [9]:
top_predictions = (
    test_data
    .merge(
        mart[
            [
                "manufacturer",
                "drug_name",
                "event_month",
                "events_count",
                "events_next_month",
            ]
        ].rename(
            columns={
                "event_month": "event_month_for_join",
                "events_count": "events_count_for_join",
            }
        ),
        left_on=["manufacturer", "drug_name", "event_month"],
        right_on=["manufacturer", "drug_name", "event_month_for_join"],
        how="left",
    )
    .sort_values("probability", ascending=False)
    .head(20)
)

top_predictions[
    [
        "manufacturer",
        "drug_name",
        "event_month",
        "events_count",
        "probability",
        "events_next_month",
        "target_next_month",
    ]
]

,manufacturer,drug_name,event_month,events_count,probability,events_next_month,target_next_month
196769,"общество с ограниченной ответственностью ""пран...",аторвастатин,2025-07-01,103.0,0.836882,0.0,0
182107,"общество с ограниченной ответственностью ""пран...",бетагистин,2025-05-01,5.0,0.825560,0.0,0
57573,"пао ""биосинтез""",эналаприл,2023-05-01,39.0,0.818484,33.0,1
147538,"открытое акционерное общество ""уралбиофарм"" (о...",аскорбиновая кислота-убф,2024-10-01,5.0,0.803360,8.0,1
182233,"общество с ограниченной ответственностью ""пран...",спиронолактон,2025-05-01,12.0,0.800431,0.0,0
75216,"общество с ограниченной ответственностью ""слав...",магния сульфат,2023-08-01,11.0,0.800431,0.0,0
32255,"ооо ""фармапол-волга""",аспаркам,2022-12-01,7.0,0.792080,4.0,1
248993,"общество с ограниченной ответственностью ""пран...",амлодипин-прана,2026-05-01,5.0,0.790479,3.0,1
237439,"общество с ограниченной ответственностью ""пран...",клопидогрел,2026-03-01,4.0,0.785039,1.0,1
96168,"акционерное общество ""алтайвитамины"" (ао ""алта...",ундевит,2023-12-01,10.0,0.784384,41.0,1


In [11]:
results.to_csv(
    REPORTS / "simple_ml_model_results.csv",
    index=False,
    encoding="cp1251",
)

top_predictions.to_csv(
    REPORTS / "simple_ml_model_top_predictions.csv",
    index=False,
    encoding="cp1251",
)

In [16]:
positive_support = classification_report(
    y_test,
    prediction,
    output_dict=True,
)["1"]["support"]

print(f"Положительных случаев в тесте: {positive_support}")
print(f"Доля положительного класса: {y_test.mean():.4%}")
print(f"ROC-AUC: {auc:.3f}")
print(f"Precision@20 ML: {model_precision:.3f}")
print(f"Precision@20 baseline: {baseline_precision:.3f}")

Положительных случаев в тесте: 814.0
Доля положительного класса: 0.3010%
ROC-AUC: 0.681
Precision@20 ML: 0.600
Precision@20 baseline: 0.400


In [17]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

threshold_results = []

for threshold in thresholds:
    threshold_prediction = (probability >= threshold).astype(int)

    report = classification_report(
        y_test,
        threshold_prediction,
        output_dict=True,
        zero_division=0,
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "precision": report["1"]["precision"],
            "recall": report["1"]["recall"],
            "f1": report["1"]["f1-score"],
            "predicted_positive": int(threshold_prediction.sum()),
        }
    )

pd.DataFrame(threshold_results)

,threshold,precision,recall,f1,predicted_positive
0,0.1,0.209929,0.363636,0.266187,1410
1,0.2,0.386364,0.146192,0.212121,308
2,0.3,0.472477,0.126536,0.199612,218
3,0.4,0.543478,0.092138,0.157563,138
4,0.5,0.597561,0.060197,0.109375,82
